# Prediction Model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D  
  
from scripts.data_operation import transform_to_daily_stays
from scripts.constants import START_DATE, END_DATE, PREDICTION_HORIZON_DAYS

from scripts.data_preparation import (
    compute_base_prices,
    compute_relative_price_index,
    compute_occupancy_rate,
    build_historical_sample,
    prepare_full_occupancy_series,
)
from scripts.demand_forecasting import fit_sarima_model, forecast_occupancy
from scripts.distribution_estimation import estimate_joint_density, extract_conditional_distribution
from scripts.pricing import select_price_quantile, recommend_apartment_prices

## Section 2.2 — Data Aggregation and Preprocessing

In [ ]:
relevant_data = pd.read_csv('data/relevant_data.csv', parse_dates=['start_date', 'end_date'])

listing_summary = relevant_data.groupby('apartment_id').agg(
    max_people_count=('beds_count', 'max'),
    num_reservations=('apartment_id', 'count'),
).reset_index()

total_capacity = listing_summary['max_people_count'].sum()

print(f"Apartments in segment : {len(listing_summary)}")
print(f"Total capacity (beds) : {total_capacity}")
listing_summary.head()

In [ ]:
# Expand booking records to one row per apartment per night
daily_stays = transform_to_daily_stays(relevant_data)
daily_stays['stay_date'] = pd.to_datetime(daily_stays['stay_date'])
daily_stays = daily_stays[
    (daily_stays['stay_date'] >= START_DATE) &
    (daily_stays['stay_date'] < END_DATE)
].reset_index(drop=True)

print(f"Records : {len(daily_stays)}")
print(f"Period  : {daily_stays['stay_date'].min().date()} -> {daily_stays['stay_date'].max().date()}")
daily_stays.head()

In [ ]:
# Base price p_bar_a: historical average price per bed for each apartment
base_prices = compute_base_prices(daily_stays)

print("Base prices per bed (p_bar_a):")
base_prices.reset_index().rename(columns={'nightly_bed_rate': 'base_price_per_bed'})

In [ ]:
# Relative price index r_t: beds-weighted average of (p_{a,t} / p_bar_a) across all apartments
relative_price_index = compute_relative_price_index(daily_stays, base_prices)

print(f"Days with r_t defined : {len(relative_price_index)}")
print(f"r_t range             : {relative_price_index.min():.3f} - {relative_price_index.max():.3f}")
print(f"r_t mean              : {relative_price_index.mean():.3f}  (should be approx 1.0)")

In [ ]:
# Occupancy rate o_t on days with at least one booking
occupancy_rate_booked = compute_occupancy_rate(daily_stays, total_capacity)

# Historical sample D = {(o_t, r_t)}
historical_sample = build_historical_sample(occupancy_rate_booked, relative_price_index)

print(f"Historical sample size : {len(historical_sample)} days")
print(f"Occupancy range        : {historical_sample['occupancy_rate'].min():.3f} - {historical_sample['occupancy_rate'].max():.3f}")
print(f"Relative price range   : {historical_sample['relative_price_index'].min():.3f} - {historical_sample['relative_price_index'].max():.3f}")

fig, axes = plt.subplots(1, 1, figsize=(10, 5))

# Primary axis: relative price index (pink)
ln1 = axes.plot(relative_price_index.index, relative_price_index.values,
                color='pink', linewidth=0.8, alpha=0.9, label='Relative Price Index $r_t$')
axes.axhline(1.0, color='grey', linestyle='--', linewidth=0.8)
axes.set_title('Relative Price Index and Occupancy Rate over Time')
axes.set_xlabel('Date')
axes.set_ylabel('Relative Price Index $r_t$')
axes.tick_params(axis='y')

# Secondary axis: occupancy rate (gray)
ax0b = axes.twinx()
ln2 = ax0b.plot(occupancy_rate_booked.index, occupancy_rate_booked.values,
                color='gray', linewidth=0.8, alpha=0.9, label='Occupancy Rate $o_t$')
ax0b.set_ylabel('Occupancy Rate $o_t$')
ax0b.set_ylim(0, 1)
ax0b.tick_params(axis='y')

lines = ln1 + ln2
axes.legend(lines, [l.get_label() for l in lines], loc='upper left', fontsize=8)

plt.tight_layout()
plt.show()

## Section 2.3 — Demand Forecasting with SARIMA

In [ ]:
# Full occupancy series including zeros on unbooked days -- required for SARIMA
full_occupancy_series = prepare_full_occupancy_series(daily_stays, total_capacity, START_DATE, END_DATE)

train_occupancy = full_occupancy_series[:-PREDICTION_HORIZON_DAYS]
test_occupancy  = full_occupancy_series[-PREDICTION_HORIZON_DAYS:]

print(f"Training : {train_occupancy.index[0].date()} -> {train_occupancy.index[-1].date()}  ({len(train_occupancy)} days)")
print(f"Test     : {test_occupancy.index[0].date()}  -> {test_occupancy.index[-1].date()}   ({len(test_occupancy)} days)")

plt.figure(figsize=(10, 3))
plt.plot(train_occupancy.index, train_occupancy.values, label='Train', linewidth=0.8, color='gray')
plt.plot(test_occupancy.index,  test_occupancy.values,  label='Test',  linewidth=0.8, color='pink')
plt.axvline(test_occupancy.index[0], color='black', linestyle='--', linewidth=0.8)
plt.title('Occupancy Rate Series -- Train / Test Split')
plt.xlabel('Date')
plt.ylabel('Occupancy Rate')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Fit SARIMA with weekly seasonality (m=7)
sarima_model = fit_sarima_model(train_occupancy, seasonal_period=7)
print(sarima_model.summary())

In [ ]:
forecasted_occupancy = forecast_occupancy(sarima_model, horizon=PREDICTION_HORIZON_DAYS)
forecasted_dates = test_occupancy.index

mae  = np.mean(np.abs(test_occupancy.values - forecasted_occupancy))
rmse = np.sqrt(np.mean((test_occupancy.values - forecasted_occupancy) ** 2))
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")

plt.figure(figsize=(10, 5))
plt.plot(train_occupancy.index[-90:], train_occupancy.values[-90:], label='Historical', linewidth=0.8, color='gray')
plt.plot(forecasted_dates, test_occupancy.values,   label='Observed',          linewidth=0.8,   color='pink' )
plt.plot(forecasted_dates, forecasted_occupancy,    label='SARIMA Forecast', linewidth=1.2,linestyle='--', color='gray')
plt.axvline(forecasted_dates[0], color='black', linestyle=':', linewidth=0.8)
plt.title('SARIMA Occupancy Forecast vs Observed')
plt.xlabel('Date')
plt.ylabel('Occupancy Rate')
plt.legend()
plt.tight_layout()
plt.show()

## Section 2.4 — Estimation of Historical Occupancy-Price Distribution

In [ ]:
_, occupancy_grid, price_grid, density_grid = estimate_joint_density(historical_sample, grid_size=100)

print(f"KDE fitted on {len(historical_sample)} observations")
print(f"Grid: {len(occupancy_grid)} occupancy x {len(price_grid)} price points")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

cf = axes[0].contourf(occupancy_grid, price_grid, density_grid, levels=20, cmap='Greys')
fig.colorbar(cf, ax=axes[0], label='Density')
axes[0].set_title('Estimated Joint Density $\hat{f}(o, r)$')
axes[0].set_xlabel('Occupancy Rate o')
axes[0].set_ylabel('Relative Price Index r')

axes[1].scatter(
    historical_sample['occupancy_rate'],
    historical_sample['relative_price_index'],
    alpha=0.3,
    s=10,
    color='gray',
    label='Observed'
)

# Contour plot (no label here)
axes[1].contour(
    occupancy_grid,
    price_grid,
    density_grid,
    levels=8,
    colors='pink',
    linewidths=0.8,
    alpha=0.6
)

# Titles and labels
axes[1].set_title('Joint Density Overlaid on Data')
axes[1].set_xlabel('Occupancy Rate o')
axes[1].set_ylabel('Relative Price Index r')

# Create proxy artist for contour
contour_proxy = Line2D([0], [0], color='pink', lw=1)

# Get existing legend handles and labels, then append contour
handles, labels = axes[1].get_legend_handles_labels()
handles.append(contour_proxy)
labels.append('Density Contours')

# Final legend
axes[1].legend(handles, labels)

plt.tight_layout()
plt.show()

## Section 2.5 — Pricing Decision

In [ ]:
# alpha: pricing quantile -- 0 = most competitive, 1 = most aggressive
PRICING_QUANTILE = 0.5

In [ ]:
# Recommendations across the full forecast horizon
records = []

for date, predicted_occ in zip(forecasted_dates, forecasted_occupancy):
    try:
        p_axis, cond_pdf = extract_conditional_distribution(
            occupancy_grid, price_grid, density_grid, predicted_occ
        )
        rel_price  = select_price_quantile(p_axis, cond_pdf, PRICING_QUANTILE)
        apt_prices = recommend_apartment_prices(rel_price, base_prices)
        records.append({
            'date': date,
            'predicted_occupancy': predicted_occ,
            'relative_price': rel_price,
            **{f'apt_{aid}_price': price for aid, price in apt_prices.items()},
        })
    except ValueError as e:
        print(f"Skipping {date.date()}: {e}")

recommendations = pd.DataFrame(records).set_index('date')

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(recommendations.index, recommendations['predicted_occupancy'], marker='o', markersize=3, color='gray')
axes[0].set_ylabel('Predicted Occupancy')
axes[0].set_title('Forecast Horizon: Predicted Occupancy and Relative Price Recommendation')
axes[0].grid(True, alpha=0.3)

axes[1].plot(recommendations.index, recommendations['relative_price'], marker='o', markersize=3, color='#88703c')
axes[1].axhline(1.0, color='#B4B4B4', linestyle='--', linewidth=0.8, label='Base level (r=1)')
axes[1].set_ylabel('Relative Price r*_t')
axes[1].set_xlabel('Date')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

recommendations[['predicted_occupancy', 'relative_price']].round(4)

In [ ]:
# Price recommendation table: nightly price per bed per apartment for the forecast horizon
apt_cols = [c for c in recommendations.columns if c.startswith('apt_') and c.endswith('_price')]
apt_ids  = [int(c.split('_')[1]) for c in apt_cols]

price_table = pd.DataFrame(index=recommendations.index)
price_table.index.name = 'date'
price_table['predicted_occupancy'] = recommendations['predicted_occupancy'].round(3)
price_table['relative_price_r*']   = recommendations['relative_price'].round(3)

for col, aid in zip(apt_cols, apt_ids):
    price_table[f'apt_{aid}'] = recommendations[col].round(0).astype(int)

print(f"Price recommendations (per bed): {len(price_table)} days  "
      f"({price_table.index[0].date()} → {price_table.index[-1].date()})")
price_table